# Laboratorio 4 · Parte 2
Pablo Cabrera · Luis Mendoza. Este notebook usa funciones de `src` y artefactos reproducibles; no contiene credenciales.

In [ ]:
from pathlib import Path
import pandas as pd
from src.parte2 import build_dataset, train_random, evaluate_validations
from src.ml_features import predictor_audit, strict_feature_columns
ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()

## Ejercicios 1–3 · Preparación, respuesta y predictores
La respuesta es `proxy_chl >= 10 µg/L`. No es una medición in situ ni de toxicidad. Se excluyen NDCI, clorofila, B04, B05, NDVI, coordenadas y lago del modelo principal.

In [ ]:
path = ROOT/'outputs/parte2/data/observations_master.parquet'
data = pd.read_parquet(path) if path.exists() else build_dataset()
display(data.shape, data.dtypes.to_frame('dtype'), data.isna().mean().to_frame('missing_fraction'))
display(data.groupby(['lake','date','alta_presencia']).size().rename('n').reset_index())
display(predictor_audit())

## Ejercicios 4–5 · Modelos y evaluación 70/30
El test común permanece intacto. La búsqueda pequeña usa PR-AUC exclusivamente dentro de entrenamiento.

In [ ]:
random_results = train_random(data)
display(random_results)

## Ejercicios 6–7 y rúbrica temporal · Validación espacial, temporal y entre lagos
Los bloques completos nunca cruzan folds. Las últimas dos fechas completas por lago forman el test temporal. Los experimentos A→B y B→A no usan el destino para ajuste.

In [ ]:
validation_results = evaluate_validations(data)
display(validation_results)
display(pd.read_csv(ROOT/'outputs/parte2/tables/temporal_split.csv'))

## Ejercicios 8–9 · Interpretabilidad y mapas
La importancia por permutación y SHAP se calculan sobre datos no usados en ajuste. Los mapas reconstruyen únicamente celdas observadas, con una escala común.

In [ ]:
# Los artefactos se generan por el pipeline principal y se muestran si existen.
from IPython.display import display, Image
for p in sorted((ROOT/'outputs/parte2/figures').glob('*.png')) + sorted((ROOT/'outputs/parte2/maps').glob('*.png')):
    display(Image(filename=str(p)))

## Ejercicio 10 · Análisis y conclusiones
Las conclusiones finales se generan desde las métricas ejecutadas. Se comparan dependencia espacial/temporal, cambio de dominio y costos de falsos negativos y falsos positivos. No se declara aptitud operativa sin validación de campo.